In [ ]:
import numpy as np
import json
import math as m
import torch

## GPT answer load

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wtype learning/final_gpt4.1_wtype_answers.json', 'r') as f:
  gpt_data = json.loads(f.read())
  f.close()

In [ ]:
gpt_input = []

for i, data_obj in enumerate(gpt_data):
  if i in [543, 678, 937, 1104]:
    continue

  input_obj = {'text': data_obj['answer']}

  if 'Label: stigma' in data_obj['answer']:
    input_obj['label'] = 1
  elif 'Label: no stigma' in data_obj['answer']:
    input_obj['label'] = 0
  else:
    print(i)

  prob_list = []
  for logp in data_obj['all_logprobs']:
    prob_list.append(m.exp(logp[0][1]))

  score = np.mean(prob_list).item()
  input_obj['prob'] = score
  gpt_input.append(input_obj)

In [ ]:
len(gpt_input)

1382

In [ ]:
count = 0
for obj in gpt_input:
  count += obj['label']

count

752

## Llama answer load

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wtype learning/final_Llama_wtype_answers.json', 'r') as f:
  llama_data = json.loads(f.read())
  f.close()

In [ ]:
llama_input = []

for i, data_obj in enumerate(llama_data):
  if i in [543, 678, 937, 1104]:
    continue

  input_obj = {'text': data_obj['answer']}

  if 'Label: stigma' in data_obj['answer']:
    input_obj['label'] = 1
  elif 'Label: no stigma' in data_obj['answer']:
    input_obj['label'] = 0
  else:
    print(i)

  score = np.mean(data_obj['probs']).item()
  input_obj['prob'] = score
  llama_input.append(input_obj)

In [ ]:
len(llama_input)

1382

In [ ]:
llama_input[0]

{'text': 'Label: stigma\nType: Perceived\nReasoning: This statement suggests that people are aware of negative views or stereotypes that may lead to hiding diabetes devices, implying a perceived stigma surrounding diabetes management.<|eot_id|>',
 'label': 1,
 'prob': 0.8536517561935797,
 'norm_prob': 0.500693573916454}

In [ ]:
count = 0
for obj in llama_input:
  count += obj['label']

count

768

## DepRoBERTa answer load

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wotype learning/all_logits_Deproberta.json', 'r') as f:
  deproberta_data = json.loads(f.read())
  f.close()

In [ ]:
deproberta_input = []

for i, data_obj in enumerate(deproberta_data):
  if i in [543, 678, 937, 1104]:
    continue

  input_obj = {}
  input_obj['label'] = torch.softmax(torch.tensor(data_obj), dim=0).argmax().item()
  input_obj['prob'] = torch.softmax(torch.tensor(data_obj), dim=0).max().item()
  deproberta_input.append(input_obj)

In [ ]:
len(deproberta_input)

1382

In [ ]:
count = 0
for obj in deproberta_input:
  count += obj['label']

count

447

# Ensemble weight computation

In [ ]:
def compute_KL(prob_list):
  return sum([x*m.log(len(prob_list)*x) for x in prob_list])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

deproberta_prob = [obj['prob'] for obj in deproberta_input]
gpt_prob = [obj['prob'] for obj in gpt_input]
llama_prob = [obj['prob'] for obj in llama_input]

scaler = MinMaxScaler()

norm_dep_prob = scaler.fit_transform(np.array(deproberta_prob).reshape(-1, 1))
norm_gpt_prob = scaler.fit_transform(np.array(gpt_prob).reshape(-1, 1))
norm_llama_prob = scaler.fit_transform(np.array(llama_prob).reshape(-1, 1))

for i in range(len(deproberta_input)):
  deproberta_input[i]['norm_prob'] = norm_dep_prob[i, 0].item()

for i in range(len(gpt_input)):
  gpt_input[i]['norm_prob'] = norm_gpt_prob[i, 0].item()

for i in range(len(llama_input)):
  llama_input[i]['norm_prob'] = norm_llama_prob[i, 0].item()

scores = [compute_KL(deproberta_prob),
          compute_KL(gpt_prob),
          compute_KL(llama_prob),]

sum_s = sum(scores)
weights = [s / sum_s for s in scores]

weights

[0.30953335003031385, 0.3379481169004467, 0.3525185330692394]

In [ ]:
norm_dep_prob[1, 0].item()

0.059228578685064814

# Sample selection loop (semi-supervised)

In [ ]:
# Normalize prob before completing this
all_pseudo_samples = []

for i, (dep, gpt, llama) in enumerate(zip(deproberta_input, gpt_input, llama_input)):
  vote_score = dep['label'] + gpt['label'] + llama['label']

  if vote_score == 0 or vote_score == 3:
    final_obj = {'label': dep['label'],
                 'text': gpt['text'] if gpt['prob'] > llama['prob'] else llama['text']}
    all_pseudo_samples.append(final_obj)
  else:
    S1 = 0
    S0 = 0

    for j, model in enumerate([dep, gpt, llama]):
      if model['label'] == 1:
        S1 += weights[j] * model['norm_prob']
      elif model['label'] == 0:
        S0 += weights[j] * model['norm_prob']

    final_label = 1 if S1 > S0 else 0
    final_obj = {'label': final_label,
                 'text': gpt['text'] if gpt['prob'] > llama['prob'] else llama['text']}
    all_pseudo_samples.append(final_obj)

In [ ]:
count = sum([d['label'] for d in all_pseudo_samples])
count

712

In [ ]:
all_pseudo_samples[0]

{'label': 1,
 'text': 'Label: stigma  \nType: Anticipated  \nReasoning: The text describes people hiding diabetes devices due to fear or expectation of being stigmatized, indicating anticipated stigma.'}

# Parse and select the final samples

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wotype learning/integrated.json', 'r') as f:
  raw_data = json.loads(f.read())
  f.close()

In [ ]:
raw_data[0]

{'input text': 'Diabetes stigma can cause people to hide visible devices like pumps or CGMs.'}

In [ ]:
len(raw_data)

1386

In [ ]:
del raw_data[543]
del raw_data[677]
del raw_data[935]
del raw_data[1101]

In [ ]:
positive_cases = []
for i in range(len(raw_data)):
  if all_pseudo_samples[i]['label'] == 1:
    data_obj = {'input text': raw_data[i]['input text'],
                'label': 'Yes stigma',
                'types': all_pseudo_samples[i]['text'].split('\n')[1].split(':')[1].strip().split(','),
                'reasoning': all_pseudo_samples[i]['text'].split('\n')[2].strip().split('Reasoning: ')[1].strip().replace('<|eot_id|>', '')}
    positive_cases.append(data_obj)

In [ ]:
positive_cases[0]

{'input text': 'Diabetes stigma can cause people to hide visible devices like pumps or CGMs.',
 'label': 'Yes stigma',
 'reasoning': 'The text discusses the impact of diabetes stigma, specifically how it leads people to hide their medical devices due to fear of judgment or discrimination. This highlights the presence and effects of stigma related to diabetes.'}

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wtype learning/corrected_positive_cases.json', 'w') as f:
  f.write(json.dumps(positive_cases))
  f.close()

# Sample selection loop (majority voting)

In [ ]:
# Normalize prob before completing this
maj_pseudo_samples = []

for i, (dep, gpt, llama) in enumerate(zip(deproberta_input, gpt_input, llama_input)):
  vote_score = dep['label'] + gpt['label'] + llama['label']

  if vote_score >= 2:
    final_obj = {'label': 1,
                 'text': gpt['text'] if gpt['prob'] > llama['prob'] else llama['text']}
  else:
    final_obj = {'label': 0,
                 'text': gpt['text'] if gpt['prob'] > llama['prob'] else llama['text']}

  maj_pseudo_samples.append(final_obj)


In [ ]:
count = sum([d['label'] for d in maj_pseudo_samples])
count

713

In [ ]:
maj_positive_cases = []
for i in range(len(raw_data)):
  if maj_pseudo_samples[i]['label'] == 1:
    data_obj = {'input text': raw_data[i]['input text'],
                'label': 'Yes stigma',
                'types': maj_pseudo_samples[i]['text'].split('\n')[1].split(':')[1].strip().split(','),
                'reasoning': maj_pseudo_samples[i]['text'].split('\n')[2].strip().split('Reasoning: ')[1].strip().replace('<|eot_id|>', '')}
    maj_positive_cases.append(data_obj)

In [ ]:
len(maj_positive_cases)

713

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wtype learning/majority_positive_cases.json', 'w') as f:
  f.write(json.dumps(maj_positive_cases))
  f.close()

# GPT samples only

In [ ]:
# Normalize prob before completing this
gpt_pseudo_samples = []

for i, (_, gpt, _) in enumerate(zip(deproberta_input, gpt_input, llama_input)):

  final_obj = {'label': gpt['label'],
               'text': gpt['text'],
  }

  gpt_pseudo_samples.append(final_obj)

In [ ]:
count = sum([d['label'] for d in gpt_pseudo_samples])
count

752

In [ ]:
gpt_positive_cases = []
for i in range(len(raw_data)):
  if gpt_pseudo_samples[i]['label'] == 1:
    data_obj = {'input text': raw_data[i]['input text'],
                'label': 'Yes stigma',
                'types': gpt_pseudo_samples[i]['text'].split('\n')[1].split(':')[1].strip().split(','),
                'reasoning': gpt_pseudo_samples[i]['text'].split('\n')[2].strip().split('Reasoning: ')[1].strip().replace('<|eot_id|>', '')}
    gpt_positive_cases.append(data_obj)

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised wtype learning/gpt_positive_cases.json', 'w') as f:
  f.write(json.dumps(gpt_positive_cases))
  f.close()